# Snake Game Using Pygame

This project implements a Snake game using the Pygame library. The player controls a snake that moves around the game window, collects apples, and increases its score and length.

The game also includes a second snake that becomes active after the player's score reaches a certain threshold. The second snake attempts to follow the apple, creating an additional challenge.

## Game Setup and Initialization

The game uses Pygame to create the display window, load images and sounds, initialize the clock, and define the initial positions of the game objects.

The player snake and the second snake are represented by lists of rectangular body segments. The apple and other game objects are represented using Pygame rectangles.

## Player Controls and Movement

The player controls the snake using the arrow keys. The snake moves continuously in its current direction, and the player can change direction while preventing immediate movement in the opposite direction.

When the snake eats an apple, its score increases and its body grows by one segment.

## Second Snake and Game Logic

A second snake is activated when the player's score exceeds 10. It attempts to move toward the apple and can also collect it.

The game includes additional objects and collision conditions. Collisions with certain objects affect the game state, while reaching the game-over condition ends the game.

## Implementation and Execution

The complete implementation is provided in one code cell. The game starts with a welcome screen, followed by the main game loop. The player can quit using the Q key or exit using the Escape key.

Place the required image and sound files in an `assets` directory before running the notebook.

In [ ]:
import os
import random
import pygame


# ============================================================
# Game Initialization
# ============================================================

pygame.init()
pygame.mixer.init()

SCREEN_WIDTH = 1000
SCREEN_HEIGHT = 1000
BLOCK_SIZE = 100
FPS = 10

screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Snake")

clock = pygame.time.Clock()
font = pygame.font.SysFont(None, 25)

red = (220, 20, 60)
white = (255, 255, 255)
background = (200, 200, 200)

ASSET_DIR = "assets"


# ============================================================
# Asset Loading
# ============================================================

def load_image(filename, size=(100, 100)):
    path = os.path.join(ASSET_DIR, filename)
    image = pygame.image.load(path).convert_alpha()
    return pygame.transform.scale(image, size)


image = load_image("circle.png")
image2 = load_image("Golden_Big_Chain_Chomp_Sprite_PM5.png")
image3 = load_image("Featured-Image.png")
image4 = load_image("super_mario_world_mushroom_sprite_by_adventuresmg64gaming-db12tjs.png")
apple_image = load_image("apple90.png")

pygame.display.set_icon(pygame.image.load(
    os.path.join(ASSET_DIR, "Golden_Big_Chain_Chomp_Sprite_PM5.png")
).convert_alpha())

EAT_SOUND_PATH = os.path.join(ASSET_DIR, "burger (1).wav")
eat_sound = pygame.mixer.Sound(EAT_SOUND_PATH)


# ============================================================
# Helper Functions
# ============================================================

def random_position():
    """Generate a random position aligned with the movement grid."""
    x = random.randrange(100, 900, BLOCK_SIZE)
    y = random.randrange(100, 900, BLOCK_SIZE)
    return x, y


def create_segment(x, y):
    """Create a rectangular body segment at the specified position."""
    rect = image.get_rect()
    rect.x = x
    rect.y = y
    return rect


def message(msg, color, y_displace=0):
    text_surface = font.render(msg, True, color)
    text_rect = text_surface.get_rect()
    text_rect.center = (SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + y_displace)
    screen.blit(text_surface, text_rect)


def draw_score(value, x=0):
    text = font.render("Score:" + str(value), True, red)
    screen.blit(text, (x, 0))


def draw_snake(body, head_image, body_image):
    """Draw the snake, using a separate image for its head."""
    for segment in body[:-1]:
        screen.blit(body_image, segment)

    if body:
        screen.blit(head_image, body[-1])


def spawn_apple(apple):
    apple.x, apple.y = random_position()


# ============================================================
# Initial Game Objects
# ============================================================

player_body = [
    create_segment(700, 400),
    create_segment(600, 400),
    create_segment(500, 400),
    create_segment(400, 400)
]

second_body = [
    create_segment(800, 500),
    create_segment(700, 500),
    create_segment(600, 500),
    create_segment(500, 500)
]

apple = apple_image.get_rect()
spawn_apple(apple)

bonus = image4.get_rect()
bonus.x, bonus.y = random_position()

obstacle = image3.get_rect()
obstacle.x, obstacle.y = 200, 200

player_score = 0
second_score = 0

# Directions: 0 = down, 1 = right, 2 = up, 3 = left
player_direction = 1
second_direction = 1

game_running = True
game_started = False


# ============================================================
# Welcome Screen
# ============================================================

screen.fill(white)
message("Welcome to Snake", red, -90)
message("Eat the Apple!", red, -40)
message("Press P to play or Q to quit!", red, -10)
pygame.display.update()

while not game_started and game_running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            game_running = False

        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_p:
                game_started = True
            elif event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                game_running = False


# ============================================================
# Player Movement
# ============================================================

def handle_player_direction():
    global player_direction

    keys = pygame.key.get_pressed()

    if keys[pygame.K_LEFT] and player_direction in (0, 2):
        player_direction = 3
    elif keys[pygame.K_RIGHT] and player_direction in (0, 2):
        player_direction = 1
    elif keys[pygame.K_DOWN] and player_direction in (1, 3):
        player_direction = 0
    elif keys[pygame.K_UP] and player_direction in (1, 3):
        player_direction = 2


def get_movement(direction):
    if direction == 0:
        return 0, BLOCK_SIZE
    elif direction == 1:
        return BLOCK_SIZE, 0
    elif direction == 2:
        return 0, -BLOCK_SIZE
    elif direction == 3:
        return -BLOCK_SIZE, 0

    return 0, 0


def move_snake(body, direction, grow=False):
    """Move a snake by adding a new head and optionally removing its tail."""
    move_x, move_y = get_movement(direction)
    new_head = create_segment(body[-1].x + move_x, body[-1].y + move_y)
    body.append(new_head)

    if not grow:
        del body[0]

    return new_head


# ============================================================
# Second Snake Movement
# ============================================================

def update_second_direction():
    global second_direction

    head = second_body[-1]
    dx = apple.x - head.x
    dy = apple.y - head.y

    # Choose a direction that moves toward the apple.
    if second_direction in (1, 3):
        if dy > 0:
            second_direction = 0
        elif dy < 0:
            second_direction = 2
        elif dx > 0:
            second_direction = 1
        elif dx < 0:
            second_direction = 3
    else:
        if dx > 0:
            second_direction = 1
        elif dx < 0:
            second_direction = 3
        elif dy > 0:
            second_direction = 0
        elif dy < 0:
            second_direction = 2


# ============================================================
# Main Game Loop
# ============================================================

while game_running and game_started:
    clock.tick(FPS)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            game_running = False

        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE or event.key == pygame.K_q:
                game_running = False

    if not game_running:
        break

    screen.fill(background)

    # Draw static objects.
    screen.blit(apple_image, apple)
    screen.blit(image3, obstacle)
    screen.blit(image4, bonus)

    # --------------------------------------------------------
    # Update Player Snake
    # --------------------------------------------------------

    handle_player_direction()
    player_head = move_snake(player_body, player_direction)

    if player_head.colliderect(apple):
        player_score += 10
        # Restore the removed tail segment to grow the snake.
        player_body.insert(0, create_segment(
            player_body[0].x, player_body[0].y
        ))
        eat_sound.play()
        spawn_apple(apple)

    if player_head.colliderect(bonus):
        player_score -= 10
        bonus.x, bonus.y = random_position()

    # --------------------------------------------------------
    # Update Second Snake
    # --------------------------------------------------------

    if player_score > 10:
        update_second_direction()
        second_head = move_snake(second_body, second_direction)

        if second_head.colliderect(apple):
            second_score += 10
            second_body.insert(0, create_segment(
                second_body[0].x, second_body[0].y
            ))
            eat_sound.play()
            spawn_apple(apple)

        if second_head.colliderect(bonus):
            bonus.x, bonus.y = random_position()

    # --------------------------------------------------------
    # Collision Detection and Game Over
    # --------------------------------------------------------

    if player_head.colliderect(obstacle) or player_score <= -50:
        screen.fill(white)
        message("Game Over!", red, -90)
        pygame.display.update()
        pygame.time.wait(1500)
        game_running = False
        break

    # --------------------------------------------------------
    # Drawing Snakes and Scores
    # --------------------------------------------------------

    draw_snake(player_body, image2, image)

    if player_score > 10:
        draw_snake(second_body, image2, image)

    draw_score(player_score, 0)

    if player_score > 10:
        draw_score(second_score, 100)

    pygame.display.update()


# ============================================================
# Quit Pygame
# ============================================================

pygame.quit()